# Pima Indians Diabetes — 딥러닝 이진 분류 실습

**데이터**: `pandas/data/diabetes.csv`
**목표**: 당뇨병 발병 여부(`Outcome`) 예측 — **이진 분류(Binary Classification)**

## 학습 흐름
1. 데이터 탐색적 분석 (EDA)
2. 시각화 분석
3. 데이터 전처리
4. 데이터 분할
5. 딥러닝 학습 (Baseline)
6. 결과 분석
7. 모델 개선
8. 개선 결과 분석

## 컬럼 설명
| 컬럼 | 의미 |
|------|------|
| Pregnancies | 임신 횟수 |
| Glucose | 포도당 수치 |
| BloodPressure | 혈압 |
| SkinThickness | 피부 두께 |
| Insulin | 인슐린 |
| BMI | 체질량지수 |
| DiabetesPedigreeFunction | 당뇨 가족력 지수 |
| Age | 나이 |
| **Outcome** | 당뇨 여부 (0/1) — 타겟 |

## 1. 데이터 탐색적 분석 (EDA)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

df = pd.read_csv('pandas/data/diabetes.csv')
print(f'데이터 shape: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
# 통계 요약
df.describe()

In [ ]:
# 타겟 분포
print('=== Outcome 분포 ===')
print(df['Outcome'].value_counts())
print(f'\n당뇨 발병률: {df["Outcome"].mean():.4f}')

In [ ]:
# 0값 확인 — Pima 데이터셋의 특성: 결측치가 0으로 기록되어 있음
# Glucose, BloodPressure, SkinThickness, Insulin, BMI는 0이 될 수 없음
zero_check = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print('=== 0값 개수 (사실상 결측치) ===')
for col in zero_check:
  cnt = (df[col] == 0).sum()
  print(f'  {col:20s}: {cnt:4d} ({cnt/len(df)*100:.1f}%)')

## 2. 시각화 분석

In [ ]:
# 1) 타겟 분포
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x='Outcome', ax=axes[0], palette=['skyblue', 'salmon'])
axes[0].set_title('당뇨 발병 인원 수')
axes[0].set_xticklabels(['정상(0)', '당뇨(1)'])

df['Outcome'].value_counts().plot.pie(
  ax=axes[1], autopct='%.1f%%', colors=['skyblue', 'salmon'],
  labels=['정상', '당뇨']
)
axes[1].set_ylabel('')
axes[1].set_title('당뇨 발병률')

plt.tight_layout()
plt.show()

In [ ]:
# 2) feature별 분포 (Outcome별 비교)
features = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
            'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, features):
  sns.histplot(data=df, x=col, hue='Outcome', kde=True, bins=25,
               ax=ax, palette=['skyblue', 'salmon'])
  ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# 3) 박스플롯으로 Outcome별 분포 비교
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, col in zip(axes.flat, features):
  sns.boxplot(data=df, x='Outcome', y=col, ax=ax, palette=['skyblue', 'salmon'])
  ax.set_title(col)

plt.tight_layout()
plt.show()

In [ ]:
# 4) 상관관계 히트맵
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('feature 상관관계')
plt.tight_layout()
plt.show()
print('→ Glucose, BMI, Age가 Outcome과 가장 상관 높음')

In [ ]:
# 5) Glucose vs BMI 산점도 (당뇨 여부별)
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x='Glucose', y='BMI', hue='Outcome',
                palette=['skyblue', 'salmon'], alpha=0.7)
plt.title('Glucose × BMI (당뇨 여부별)')
plt.tight_layout()
plt.show()

## 3. 데이터 전처리

### 핵심 처리
- **0값 → NaN으로 변환 후 중앙값으로 대체** (Glucose, BP, Skin, Insulin, BMI)
- **StandardScaler 스케일링**

Pima 데이터셋의 가장 큰 특징은 **결측치가 0으로 기록**되어 있다는 점입니다.
(혈압이 0인 사람은 없으므로 명백한 결측치)

In [ ]:
df_p = df.copy()

# 0이 결측치인 컬럼 → NaN → 중앙값 대체
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for col in zero_cols:
  df_p[col] = df_p[col].replace(0, np.nan)
  df_p[col] = df_p[col].fillna(df_p[col].median())

print(f'결측치 합계: {df_p.isnull().sum().sum()}')
print(f'전처리 후 shape: {df_p.shape}')

## 4. 데이터 분할
- Train / Test = 80 / 20
- `stratify`로 클래스 비율 유지
- StandardScaler 적용

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_p.drop('Outcome', axis=1)
y = df_p['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)
y_train = y_train.values.astype(np.float32)
y_test = y_test.values.astype(np.float32)

print(f'X_train: {X_train_s.shape}, X_test: {X_test_s.shape}')
print(f'train 발병률: {y_train.mean():.4f}, test 발병률: {y_test.mean():.4f}')

## 5. 딥러닝 학습 — Baseline
- 은닉층 2개 (Dense 32 → 16, ReLU)
- 출력층: `Dense(1, sigmoid)`
- loss: `binary_crossentropy`

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential, layers

tf.random.set_seed(42)
np.random.seed(42)

input_dim = X_train_s.shape[1]

baseline = Sequential([
  layers.Dense(32, activation='relu', input_shape=(input_dim,)),
  layers.Dense(16, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

baseline.compile(
  optimizer='adam',
  loss='binary_crossentropy',
  metrics=['accuracy']
)
baseline.summary()

In [ ]:
history_base = baseline.fit(
  X_train_s, y_train,
  validation_split=0.2,
  epochs=100,
  batch_size=32,
  verbose=0
)
print(f'학습 완료 (epoch={len(history_base.history["loss"])})')

## 6. 결과 분석 (Baseline)
- 학습 곡선
- 평가 지표 (Accuracy, Precision, Recall, F1, AUC)
- 혼동 행렬

In [ ]:
# 학습 곡선 함수 (재사용)
def plot_history(history, title='Learning Curve'):
  fig, axes = plt.subplots(1, 2, figsize=(13, 4))
  axes[0].plot(history.history['loss'], label='train')
  axes[0].plot(history.history['val_loss'], label='val')
  axes[0].set_title(f'{title} — Loss')
  axes[0].set_xlabel('Epoch')
  axes[0].legend()
  axes[1].plot(history.history['accuracy'], label='train')
  axes[1].plot(history.history['val_accuracy'], label='val')
  axes[1].set_title(f'{title} — Accuracy')
  axes[1].set_xlabel('Epoch')
  axes[1].legend()
  plt.tight_layout()
  plt.show()

plot_history(history_base, 'Baseline')

In [ ]:
from sklearn.metrics import (
  accuracy_score, precision_score, recall_score, f1_score,
  roc_auc_score, confusion_matrix, classification_report
)

results = {}

def evaluate_clf(name, model, X_te, y_te):
  prob = model.predict(X_te, verbose=0).flatten()
  pred = (prob >= 0.5).astype(int)
  metrics = {
    'Accuracy': accuracy_score(y_te, pred),
    'Precision': precision_score(y_te, pred),
    'Recall': recall_score(y_te, pred),
    'F1': f1_score(y_te, pred),
    'AUC': roc_auc_score(y_te, prob),
  }
  results[name] = metrics
  print(f'=== [{name}] ===')
  for k, v in metrics.items():
    print(f'  {k:10s}: {v:.4f}')
  return pred, prob

pred_base, prob_base = evaluate_clf('Baseline', baseline, X_test_s, y_test)

In [ ]:
cm = confusion_matrix(y_test, pred_base)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['정상', '당뇨'], yticklabels=['정상', '당뇨'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Baseline 혼동 행렬')
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred_base, target_names=['정상', '당뇨']))

## 7. 모델 개선

### 개선 포인트
| 항목 | Baseline | Improved |
|------|---------|----------|
| 모델 깊이 | 32 → 16 | **64 → 32 → 16** |
| BatchNormalization | 없음 | **추가** |
| Dropout | 없음 | **추가** |
| Optimizer | adam (default) | **Adam(lr=0.001)** |
| EarlyStopping | 없음 | **추가** |
| ReduceLROnPlateau | 없음 | **추가** |

In [ ]:
from tensorflow.keras import callbacks, optimizers

tf.random.set_seed(42)
np.random.seed(42)

improved = Sequential([
  layers.Dense(64, input_shape=(input_dim,)),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.Dropout(0.3),

  layers.Dense(32),
  layers.BatchNormalization(),
  layers.Activation('relu'),
  layers.Dropout(0.2),

  layers.Dense(16, activation='relu'),
  layers.Dense(1, activation='sigmoid')
])

improved.compile(
  optimizer=optimizers.Adam(learning_rate=0.001),
  loss='binary_crossentropy',
  metrics=['accuracy']
)
improved.summary()

In [ ]:
early_stop = callbacks.EarlyStopping(
  monitor='val_loss', patience=20, restore_best_weights=True
)
reduce_lr = callbacks.ReduceLROnPlateau(
  monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5
)

history_imp = improved.fit(
  X_train_s, y_train,
  validation_split=0.2,
  epochs=300,
  batch_size=32,
  callbacks=[early_stop, reduce_lr],
  verbose=0
)
print(f'학습 완료 (epoch={len(history_imp.history["loss"])})')

## 8. 개선 결과 분석 및 비교

In [ ]:
plot_history(history_imp, 'Improved')

In [ ]:
pred_imp, prob_imp = evaluate_clf('Improved', improved, X_test_s, y_test)

In [ ]:
# 두 모델 혼동 행렬 비교
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, name, pred in zip(axes, ['Baseline', 'Improved'], [pred_base, pred_imp]):
  cm = confusion_matrix(y_test, pred)
  sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
              xticklabels=['정상', '당뇨'], yticklabels=['정상', '당뇨'])
  ax.set_title(name)
  ax.set_xlabel('Predicted')
  ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

In [ ]:
result_df = pd.DataFrame(results).T
print('=== 모델 성능 비교 ===')
print(result_df.round(4))

ax = result_df.plot(kind='bar', figsize=(10, 5), colormap='Set2')
ax.set_title('Baseline vs Improved')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.legend(loc='lower right')
for c in ax.containers:
  ax.bar_label(c, fmt='%.3f', fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve 비교
from sklearn.metrics import roc_curve

plt.figure(figsize=(7, 6))
for name, prob in [('Baseline', prob_base), ('Improved', prob_imp)]:
  fpr, tpr, _ = roc_curve(y_test, prob)
  auc = roc_auc_score(y_test, prob)
  plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve 비교')
plt.legend()
plt.tight_layout()
plt.show()

## 정리

### Pima Diabetes 데이터의 핵심 포인트
- **결측치가 0으로 기록**되어 있다는 점이 가장 중요
  - Glucose=0, BMI=0 등은 명백한 결측치
  - 0을 그대로 학습시키면 성능이 크게 떨어짐
- 클래스 불균형 (정상 65% / 당뇨 35%) — 약한 불균형
- Glucose, BMI, Age가 가장 중요한 feature

### 의료 도메인에서의 평가 우선순위
- **Recall 우선**: 당뇨 환자를 놓치면 안 되므로
- 단순 Accuracy보다 **F1, AUC, Recall**을 중시
- 필요 시 threshold를 0.4~0.45로 낮춰 Recall 추가 향상

### 시험 출제 포인트
- 이진 분류: `Dense(1, sigmoid)` + `binary_crossentropy`
- 학습 곡선에서 train/val 차이로 과적합 진단
- 혼동 행렬 해석 (TP/FP/TN/FN)
- ROC Curve와 AUC의 의미